In [ ]:
import pandas as pd
import os
import logging

Run_name = 'Run_39'

# Set up logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler("tropomi_merging.log"),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger()

def merge_and_save_dataframes(Run_name):
    # Path to the original file with t2m data
    original_file_path = '/net/fs06/d3/rzhuang/TROPOMI_US/data/processed_valid_tropomi_emissions_with_qa_with_t2m.csv'
    
    # Path to the new processed data
    new_file_path = f'../data/{Run_name}/valid_tropomi_emissions_with_qa.csv'
    
    # Output directory and filename
    output_dir = os.path.dirname(new_file_path)
    output_path = os.path.join(output_dir, 'updated_tropomi_emissions.csv')
    
    logger.info(f"Loading original data from: {original_file_path}")
    original_df = pd.read_csv(original_file_path)
    logger.info(f"Loaded {len(original_df)} rows from original data")
    
    logger.info(f"Loading new processed data from: {new_file_path}")
    new_df = pd.read_csv(new_file_path)
    logger.info(f"Loaded {len(new_df)} rows from new processed data")
    
    # Create a merged dataframe based on identifying columns
    # Assuming location, latitude, longitude, and utc_time can uniquely identify a row
    logger.info("Creating mapping between original and new data")
    
    # Create matching keys in both dataframes
    original_df['match_key'] = original_df['location'].astype(str) + '_' + \
                               original_df['latitude'].astype(str) + '_' + \
                               original_df['longitude'].astype(str) + '_' + \
                               original_df['utc_time'].astype(str)
    
    new_df['match_key'] = new_df['location'].astype(str) + '_' + \
                          new_df['latitude'].astype(str) + '_' + \
                          new_df['longitude'].astype(str) + '_' + \
                          new_df['utc_time'].astype(str)
    
    # Create a dictionary mapping from match_key to new values
    new_values_dict = {}
    for idx, row in new_df.iterrows():
        new_values_dict[row['match_key']] = {
            'plume_label': row['plume_label'],
            'no2_std_50km': row['no2_std_50km'], 
            # Add any other columns you want to update here
        }
    
    # Count matches for reporting
    match_count = 0
    
    # Update the original dataframe with new values
    logger.info("Updating original dataframe with new values")
    for idx, row in original_df.iterrows():
        match_key = row['match_key']
        if match_key in new_values_dict:
            match_count += 1
            original_df.at[idx, 'plume_label'] = new_values_dict[match_key]['plume_label']
            original_df.at[idx, 'no2_std_50km'] = new_values_dict[match_key]['no2_std_50km']
            # Update any other columns here
    
    logger.info(f"Updated {match_count} rows out of {len(original_df)} original rows")
    
    # Drop the no2_var_50km column and the temporary match_key column
    logger.info("Dropping unwanted columns")
    columns_to_drop = ['no2_var_50km', 'match_key']
    original_df = original_df.drop(columns=columns_to_drop, errors='ignore')
    
    # Save the updated dataframe
    logger.info(f"Saving updated dataframe to: {output_path}")
    original_df.to_csv(output_path, index=False)
    logger.info(f"Successfully saved {len(original_df)} rows to {output_path}")
    
    return output_path

if __name__ == "__main__":
    logger.info("=" * 80)
    logger.info("Starting TROPOMI data merge")
    logger.info("=" * 80)
    
    output_file = merge_and_save_dataframes(Run_name)
    
    logger.info("=" * 80)
    logger.info(f"Merge completed. Output saved to: {output_file}")
    logger.info("=" * 80)

2025-05-20 16:18:26,802 - INFO - ================================================================================
2025-05-20 16:18:26,803 - INFO - Starting TROPOMI data merge
2025-05-20 16:18:26,819 - INFO - ================================================================================
2025-05-20 16:18:26,820 - INFO - Loading original data from: /net/fs06/d3/rzhuang/TROPOMI_US/data/processed_valid_tropomi_emissions_with_qa_with_t2m.csv


In [60]:
import pandas as pd
import os
import logging

Run_name = 'Run_9'

# load the full power_plants DF
power_plants = pd.read_csv('/net/fs06/d3/rzhuang/TROPOMI_US/data/power_plants_with_combined_nearby_stats_parallel_debug.csv')

# filter: no other plants within 20 km, but at least one city within 20 km
mask = (
    (power_plants['nearby_plants_count_20km'] == 0) &
    (power_plants['nearby_cities_count_20km'] == 0)
)

# apply filter and take the first 100 (or, if you have a ranking metric, sort before .head)
top100 = power_plants.loc[mask].head(100)
top50 = power_plants.loc[mask].head(50)
top20 = power_plants.loc[mask].head(20)

tropomi_combined_dropna = pd.read_csv(
    f'/net/fs06/d3/rzhuang/TROPOMI_US/data/{Run_name}/updated_tropomi_emissions.csv',
)

import pandas as pd

# identify facilities with ≥60 records
counts = tropomi_combined_dropna['facility_id'].value_counts()
valid_ids = counts[counts >= 60].index

# filter & sample 60 per facility with a fixed seed
tropomi_sampled = (
    tropomi_combined_dropna[tropomi_combined_dropna['facility_id'].isin(valid_ids)]
    .groupby('facility_id', group_keys=False)
    .apply(lambda g: g.sample(n=60, random_state=42))
    .reset_index(drop=True)
)

# tropomi_combined_dropna is unchanged; tropomi_sampled holds your reproducible subset
print(tropomi_sampled.shape)

filtered_100 = tropomi_sampled[tropomi_sampled['facility_id'].isin(top100['Facility ID'])]
filtered_50 = tropomi_sampled[tropomi_sampled['facility_id'].isin(top50['Facility ID'])]
filtered_20 = tropomi_sampled[tropomi_sampled['facility_id'].isin(top20['Facility ID'])]

(28740, 55)


/tmp/ipykernel_2386534/3717153783.py:35: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(n=60, random_state=42))


In [61]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from imblearn.over_sampling import RandomOverSampler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score, classification_report, cohen_kappa_score
import matplotlib.pyplot as pnotlt

# tropomi_combined_dropna['wind_speed'] = np.sqrt(tropomi_combined_dropna['wind_u']**2 + tropomi_combined_dropna['wind_v']**2)

# print(X)
X = tropomi_sampled.iloc[:,10:]
# X = df.loc['surface_albedo']

y = tropomi_sampled["plume_label"].astype(bool)

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# # Balance training data
ros = RandomOverSampler(random_state=42)
X_train_bal, y_train_bal = ros.fit_resample(X_train, y_train)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_bal)
X_test_scaled = scaler.transform(X_test)

# Train logistic regression
model = LogisticRegression(max_iter=1000, random_state=42)
# model.fit(X_train_scaled, y_train_bal)
model.fit(X_train_scaled, y_train_bal)

# Evaluate model
y_pred = model.predict(X_test_scaled)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("AUC:", roc_auc_score(y_test, model.predict_proba(X_test_scaled)[:, 1]))
print("Classification Report:\n", classification_report(y_test, y_pred))
print("Kappa:", cohen_kappa_score(y_test, y_pred))


Accuracy: 0.6924147529575505
Precision: 0.5715764056571232
Recall: 0.7590471827759964
F1 Score: 0.6521054702872885
Confusion Matrix:
 [[2323 1242]
 [ 526 1657]]
AUC: 0.7724890602443079
Classification Report:
               precision    recall  f1-score   support

       False       0.82      0.65      0.72      3565
        True       0.57      0.76      0.65      2183

    accuracy                           0.69      5748
   macro avg       0.69      0.71      0.69      5748
weighted avg       0.72      0.69      0.70      5748

Kappa: 0.38611341027848833


In [62]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from imblearn.over_sampling import RandomOverSampler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score, classification_report, cohen_kappa_score
import matplotlib.pyplot as pnotlt

# tropomi_combined_dropna['wind_speed'] = np.sqrt(tropomi_combined_dropna['wind_u']**2 + tropomi_combined_dropna['wind_v']**2)

# print(X)
X = filtered_100.iloc[:,10:]
# X = df.loc['surface_albedo']

y = filtered_100["plume_label"].astype(bool)

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# # Balance training data
ros = RandomOverSampler(random_state=42)
X_train_bal, y_train_bal = ros.fit_resample(X_train, y_train)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_bal)
X_test_scaled = scaler.transform(X_test)

# Train logistic regression
model = LogisticRegression(max_iter=1000, random_state=42)
# model.fit(X_train_scaled, y_train_bal)
model.fit(X_train_scaled, y_train_bal)

# Evaluate model
y_pred = model.predict(X_test_scaled)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("AUC:", roc_auc_score(y_test, model.predict_proba(X_test_scaled)[:, 1]))
print("Classification Report:\n", classification_report(y_test, y_pred))
print("Kappa:", cohen_kappa_score(y_test, y_pred))


Accuracy: 0.6785714285714286
Precision: 0.7793880837359098
Recall: 0.6675862068965517
F1 Score: 0.7191679049034175
Confusion Matrix:
 [[314 137]
 [241 484]]
AUC: 0.7296398807248261
Classification Report:
               precision    recall  f1-score   support

       False       0.57      0.70      0.62       451
        True       0.78      0.67      0.72       725

    accuracy                           0.68      1176
   macro avg       0.67      0.68      0.67      1176
weighted avg       0.70      0.68      0.68      1176

Kappa: 0.34862538574480617


In [63]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from imblearn.over_sampling import RandomOverSampler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score, classification_report, cohen_kappa_score
import matplotlib.pyplot as pnotlt

# tropomi_combined_dropna['wind_speed'] = np.sqrt(tropomi_combined_dropna['wind_u']**2 + tropomi_combined_dropna['wind_v']**2)

# print(X)
X = filtered_50.iloc[:,10:]
# X = df.loc['surface_albedo']

y = filtered_50["plume_label"].astype(bool)

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# # Balance training data
ros = RandomOverSampler(random_state=42)
X_train_bal, y_train_bal = ros.fit_resample(X_train, y_train)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_bal)
X_test_scaled = scaler.transform(X_test)

# Train logistic regression
model = LogisticRegression(max_iter=1000, random_state=42)
# model.fit(X_train_scaled, y_train_bal)
model.fit(X_train_scaled, y_train_bal)

# Evaluate model
y_pred = model.predict(X_test_scaled)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("AUC:", roc_auc_score(y_test, model.predict_proba(X_test_scaled)[:, 1]))
print("Classification Report:\n", classification_report(y_test, y_pred))
print("Kappa:", cohen_kappa_score(y_test, y_pred))


Accuracy: 0.6805555555555556
Precision: 0.8407079646017699
Recall: 0.6867469879518072
F1 Score: 0.7559681697612732
Confusion Matrix:
 [[107  54]
 [130 285]]
AUC: 0.7362268951582729
Classification Report:
               precision    recall  f1-score   support

       False       0.45      0.66      0.54       161
        True       0.84      0.69      0.76       415

    accuracy                           0.68       576
   macro avg       0.65      0.68      0.65       576
weighted avg       0.73      0.68      0.69       576

Kappa: 0.3069951743889522


In [64]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from imblearn.over_sampling import RandomOverSampler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score, classification_report, cohen_kappa_score
import matplotlib.pyplot as pnotlt

# tropomi_combined_dropna['wind_speed'] = np.sqrt(tropomi_combined_dropna['wind_u']**2 + tropomi_combined_dropna['wind_v']**2)

# print(X)
X = filtered_20.iloc[:,10:]
# X = df.loc['surface_albedo']

y = filtered_20["plume_label"].astype(bool)

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# # Balance training data
ros = RandomOverSampler(random_state=42)
X_train_bal, y_train_bal = ros.fit_resample(X_train, y_train)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_bal)
X_test_scaled = scaler.transform(X_test)

# Train logistic regression
model = LogisticRegression(max_iter=1000, random_state=42)
# model.fit(X_train_scaled, y_train_bal)
model.fit(X_train_scaled, y_train_bal)

# Evaluate model
y_pred = model.predict(X_test_scaled)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("AUC:", roc_auc_score(y_test, model.predict_proba(X_test_scaled)[:, 1]))
print("Classification Report:\n", classification_report(y_test, y_pred))
print("Kappa:", cohen_kappa_score(y_test, y_pred))


Accuracy: 0.7041666666666667
Precision: 0.8385093167701864
Recall: 0.75
F1 Score: 0.7917888563049853
Confusion Matrix:
 [[ 34  26]
 [ 45 135]]
AUC: 0.7125
Classification Report:
               precision    recall  f1-score   support

       False       0.43      0.57      0.49        60
        True       0.84      0.75      0.79       180

    accuracy                           0.70       240
   macro avg       0.63      0.66      0.64       240
weighted avg       0.74      0.70      0.72       240

Kappa: 0.2864321608040201
